# 🧪 Actividad Práctica — TDD · SOLUCIÓN COMPLETA

**Curso:** Pruebas de Software · 7mo Semestre  
**Tema:** TDD — Ciclo Red → Green → Refactor  
**Valle del Software**

> ⚠️ **Documento de referencia docente.** Contiene la solución completa con cada fase del ciclo documentada.

---

## ⚙️ Setup

In [ ]:
!pip install pytest ipytest --quiet

import ipytest
ipytest.autoconfig()

print("✅ Setup listo.")

---

# 🔴 FASE RED — Escribir todos los tests primero

Los tests se escriben **antes** de cualquier implementación. Al ejecutar esta celda todos deben fallar con `NameError: name 'validar_contrasena' is not defined`.

In [ ]:
%%ipytest -v

# ──────────────────────────────────────────────────────────────
#  FASE RED: todos estos tests deben FALLAR al ejecutarse aquí
#  porque validar_contrasena todavía no existe.
# ──────────────────────────────────────────────────────────────

def test_contrasena_valida():
    """
    DADO:    'Secure#123' cumple todas las reglas
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=True y errores=[]
    """
    r = validar_contrasena('Secure#123')
    assert r['es_valida'] == True
    assert r['errores'] == []


def test_contrasena_muy_corta():
    """
    DADO:    'Ab1!' tiene solo 4 caracteres
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay error de longitud
    """
    r = validar_contrasena('Ab1!')
    assert r['es_valida'] == False
    assert any('8' in e or 'longitud' in e.lower() or 'minimo' in e.lower()
               for e in r['errores'])


def test_contrasena_sin_mayusculas():
    """
    DADO:    'secure#123' no tiene mayúsculas
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay error de mayúsculas
    """
    r = validar_contrasena('secure#123')
    assert r['es_valida'] == False
    assert any('mayusc' in e.lower() for e in r['errores'])


def test_contrasena_sin_numeros():
    """
    DADO:    'Secure#abc' no tiene números
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay error de números
    """
    r = validar_contrasena('Secure#abc')
    assert r['es_valida'] == False
    assert any('numero' in e.lower() or 'dígito' in e.lower() or 'digit' in e.lower()
               for e in r['errores'])


def test_contrasena_sin_especiales():
    """
    DADO:    'Secure123' no tiene caracteres especiales
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay error de especiales
    """
    r = validar_contrasena('Secure123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])


def test_contrasena_multiples_errores():
    """
    DADO:    'abc' viola longitud, mayúsculas, números y especiales
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay al menos 3 errores
    """
    r = validar_contrasena('abc')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 3, f"Errores obtenidos: {r['errores']}"


# Resultado esperado en FASE RED:
# ERROR - NameError: name 'validar_contrasena' is not defined

---

# 🟢 FASE GREEN — Implementación mínima

Se escribe el código más simple posible para que **todos** los tests pasen.  
Todavía no se optimiza nada — eso va en el Refactor.

In [ ]:
import re

def validar_contrasena(password: str) -> dict:
    """Valida una contraseña — implementación mínima (fase GREEN)."""
    errores = []

    # Regla 1: mínimo 8 caracteres
    if len(password) < 8:
        errores.append('Minimo 8 caracteres requeridos')

    # Regla 2: al menos una mayúscula
    if not re.search(r'[A-Z]', password):
        errores.append('Debe tener al menos una mayuscula')

    # Regla 3: al menos una minúscula
    if not re.search(r'[a-z]', password):
        errores.append('Debe tener al menos una minuscula')

    # Regla 4: al menos un número
    if not re.search(r'\d', password):
        errores.append('Debe tener al menos un numero')

    # Regla 5: al menos un carácter especial
    if not re.search(r'[!@#$%^&*]', password):
        errores.append('Debe tener al menos un caracter especial (!@#$%^&*)')

    return {
        'es_valida': len(errores) == 0,
        'errores': errores
    }

print('✅ Implementación GREEN cargada.')

In [ ]:
%%ipytest -v

# ──────────────────────────────────────────────────────────────
#  FASE GREEN: los 6 tests deben pasar ahora
# ──────────────────────────────────────────────────────────────

def test_contrasena_valida():
    r = validar_contrasena('Secure#123')
    assert r['es_valida'] == True
    assert r['errores'] == []

def test_contrasena_muy_corta():
    r = validar_contrasena('Ab1!')
    assert r['es_valida'] == False
    assert any('8' in e or 'longitud' in e.lower() or 'minimo' in e.lower()
               for e in r['errores'])

def test_contrasena_sin_mayusculas():
    r = validar_contrasena('secure#123')
    assert r['es_valida'] == False
    assert any('mayusc' in e.lower() for e in r['errores'])

def test_contrasena_sin_numeros():
    r = validar_contrasena('Secure#abc')
    assert r['es_valida'] == False
    assert any('numero' in e.lower() or 'dígito' in e.lower() or 'digit' in e.lower()
               for e in r['errores'])

def test_contrasena_sin_especiales():
    r = validar_contrasena('Secure123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])

def test_contrasena_multiples_errores():
    r = validar_contrasena('abc')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 3, f"Errores obtenidos: {r['errores']}"

# Resultado esperado: 6 passed ✅

---

# 🟣 FASE REFACTOR — Mejorar el código sin romper los tests

La implementación GREEN funciona pero tiene código repetitivo (`if not re.search... errores.append...` cinco veces).  
El refactor elimina esa duplicación usando una lista de reglas declarativas.

**Cambios aplicados:**
- Cada regla se define como una tupla `(condicion_de_error, mensaje)`
- Se extraen constantes al módulo
- Se añaden `type hints` completos y docstring detallado
- La lógica principal queda en una sola línea (list comprehension)

In [ ]:
import re
from typing import List

# ── Constantes ──────────────────────────────────────────────
LONGITUD_MINIMA = 8
CARACTERES_ESPECIALES = r'[!@#$%^&*]'


# ── Funciones privadas de validación ────────────────────────

def _tiene_longitud_minima(password: str) -> bool:
    return len(password) >= LONGITUD_MINIMA

def _tiene_mayuscula(password: str) -> bool:
    return bool(re.search(r'[A-Z]', password))

def _tiene_minuscula(password: str) -> bool:
    return bool(re.search(r'[a-z]', password))

def _tiene_numero(password: str) -> bool:
    return bool(re.search(r'\d', password))

def _tiene_caracter_especial(password: str) -> bool:
    return bool(re.search(CARACTERES_ESPECIALES, password))


# ── Función pública ──────────────────────────────────────────

def validar_contrasena(password: str) -> dict:
    """
    Valida una contraseña según las reglas de seguridad.

    Args:
        password: La contraseña a validar.

    Returns:
        dict con:
          - 'es_valida' (bool): True si cumple todas las reglas.
          - 'errores' (list[str]): un mensaje por cada regla violada.
                                   Lista vacía si la contraseña es válida.

    Example:
        >>> validar_contrasena('Secure#123')
        {'es_valida': True, 'errores': []}

        >>> validar_contrasena('abc')
        {'es_valida': False, 'errores': ['Minimo 8 caracteres requeridos', ...]}
    """
    reglas: List[tuple] = [
        # (función que verifica la regla, mensaje si falla)
        (_tiene_longitud_minima,      f'Minimo {LONGITUD_MINIMA} caracteres requeridos'),
        (_tiene_mayuscula,            'Debe contener al menos una letra mayuscula'),
        (_tiene_minuscula,            'Debe contener al menos una letra minuscula'),
        (_tiene_numero,               'Debe contener al menos un numero'),
        (_tiene_caracter_especial,    'Debe contener al menos un caracter especial (!@#$%^&*)'),
    ]

    errores = [
        mensaje
        for verificar, mensaje in reglas
        if not verificar(password)
    ]

    return {
        'es_valida': len(errores) == 0,
        'errores': errores
    }


print('✅ Implementación REFACTORIZADA cargada.')

In [ ]:
%%ipytest -v

# ──────────────────────────────────────────────────────────────
#  FASE REFACTOR: los 6 tests deben seguir en verde
#  Si alguno falla, el refactor introdujo un bug.
# ──────────────────────────────────────────────────────────────

def test_contrasena_valida():
    r = validar_contrasena('Secure#123')
    assert r['es_valida'] == True
    assert r['errores'] == []

def test_contrasena_muy_corta():
    r = validar_contrasena('Ab1!')
    assert r['es_valida'] == False
    assert any('8' in e or 'longitud' in e.lower() or 'minimo' in e.lower()
               for e in r['errores'])

def test_contrasena_sin_mayusculas():
    r = validar_contrasena('secure#123')
    assert r['es_valida'] == False
    assert any('mayusc' in e.lower() for e in r['errores'])

def test_contrasena_sin_numeros():
    r = validar_contrasena('Secure#abc')
    assert r['es_valida'] == False
    assert any('numero' in e.lower() or 'dígito' in e.lower() or 'digit' in e.lower()
               for e in r['errores'])

def test_contrasena_sin_especiales():
    r = validar_contrasena('Secure123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])

def test_contrasena_multiples_errores():
    r = validar_contrasena('abc')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 3

# Resultado esperado: 6 passed ✅ — el refactor no rompió nada.

---

# 🔍 Demo interactiva — prueba manual de la función

In [ ]:
# ── Prueba manual con distintos casos ───────────────────────

casos = [
    'Secure#123',    # válida
    'abc',           # múltiples errores
    'secure#123',    # sin mayúscula
    'SECURE#123',    # sin minúscula
    'Secure#abc',    # sin número
    'Secure123',     # sin especial
    'Ab1!',          # muy corta
]

print(f"{'Contraseña':<20} {'Válida':<8} Errores")
print('-' * 70)
for caso in casos:
    r = validar_contrasena(caso)
    icono = '✅' if r['es_valida'] else '❌'
    errores_str = ' | '.join(r['errores']) if r['errores'] else '—'
    print(f"{caso:<20} {icono:<8} {errores_str}")

---

# 🏆 Desafío extra — solución

Implementación de los 3 casos adicionales con TDD.

In [ ]:
%%ipytest -v

# ── Tests del desafío extra (RED primero) ───────────────────

def test_contrasena_vacia():
    """
    DADO:    una cadena vacía
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False y hay al menos un error (sin excepción)
    """
    r = validar_contrasena('')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 1


def test_espacios_no_cuentan_como_especiales():
    """
    DADO:    'Secure 123' tiene espacio pero no !@#$%^&*
    CUANDO:  llamo a validar_contrasena
    ENTONCES: es_valida=False — el espacio NO cuenta como especial
    """
    r = validar_contrasena('Secure 123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])


def test_generar_reporte_valida():
    reporte = generar_reporte('Secure#123')
    assert 'VÁLIDA' in reporte.upper() or 'valida' in reporte.lower()


def test_generar_reporte_invalida_incluye_errores():
    reporte = generar_reporte('abc')
    assert 'INVÁLIDA' in reporte.upper() or 'invalida' in reporte.lower()
    assert 'Minimo' in reporte or 'minimo' in reporte.lower()

In [ ]:
# ── Implementación GREEN del desafío extra ──────────────────
# validar_contrasena ya maneja cadena vacía correctamente
# (len('') < 8, y ningún re.search encuentra nada → 4 errores).
# Los espacios tampoco están en [!@#$%^&*], por lo que tampoco
# cuentan. Ambos casos ya pasan sin cambios adicionales.

def generar_reporte(password: str) -> str:
    """
    Genera un reporte legible del resultado de la validación.

    Args:
        password: La contraseña a evaluar.

    Returns:
        String multilínea con el resumen de la validación.
    """
    resultado = validar_contrasena(password)
    lineas = [
        '=' * 45,
        f'  Contraseña: {"*" * len(password)}',
        f'  Estado:     {"✅ VÁLIDA" if resultado["es_valida"] else "❌ INVÁLIDA"}',
        '=' * 45,
    ]
    if resultado['errores']:
        lineas.append('  Errores encontrados:')
        for i, error in enumerate(resultado['errores'], 1):
            lineas.append(f'    {i}. {error}')
    else:
        lineas.append('  La contraseña cumple todos los requisitos.')
    lineas.append('=' * 45)
    return '\n'.join(lineas)


print(generar_reporte('Secure#123'))
print()
print(generar_reporte('abc'))

In [ ]:
%%ipytest -v

# ── Suite completa: 6 originales + 4 del desafío = 10 tests ─

def test_contrasena_valida():
    r = validar_contrasena('Secure#123')
    assert r['es_valida'] == True and r['errores'] == []

def test_contrasena_muy_corta():
    r = validar_contrasena('Ab1!')
    assert r['es_valida'] == False
    assert any('8' in e or 'minimo' in e.lower() for e in r['errores'])

def test_contrasena_sin_mayusculas():
    r = validar_contrasena('secure#123')
    assert r['es_valida'] == False
    assert any('mayusc' in e.lower() for e in r['errores'])

def test_contrasena_sin_numeros():
    r = validar_contrasena('Secure#abc')
    assert r['es_valida'] == False
    assert any('numero' in e.lower() for e in r['errores'])

def test_contrasena_sin_especiales():
    r = validar_contrasena('Secure123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])

def test_contrasena_multiples_errores():
    r = validar_contrasena('abc')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 3

# ── Desafío extra ────────────────────────────────────────────

def test_contrasena_vacia():
    r = validar_contrasena('')
    assert r['es_valida'] == False
    assert len(r['errores']) >= 1

def test_espacios_no_cuentan_como_especiales():
    r = validar_contrasena('Secure 123')
    assert r['es_valida'] == False
    assert any('especial' in e.lower() for e in r['errores'])

def test_generar_reporte_valida():
    reporte = generar_reporte('Secure#123')
    assert 'VÁLIDA' in reporte.upper() or 'valida' in reporte.lower()

def test_generar_reporte_invalida_incluye_errores():
    reporte = generar_reporte('abc')
    assert 'INVÁLIDA' in reporte.upper() or 'invalida' in reporte.lower()
    assert 'Minimo' in reporte or 'minimo' in reporte.lower()

# Resultado esperado: 10 passed ✅

---

## 📊 Resumen del ciclo aplicado

| Fase | Qué ocurrió |
|------|-------------|
| 🔴 **RED** | 6 tests escritos — todos fallan con `NameError` porque la función no existe |
| 🟢 **GREEN** | Implementación con 5 `if` directos — mínima, sin optimizar — todos los tests pasan |
| 🟣 **REFACTOR** | Se extrae cada regla a función privada, se usa list comprehension — los 6 tests siguen en verde |
| 🏆 **EXTRA** | 4 tests adicionales escritos — `generar_reporte` implementada — 10 tests pasan en total |

---

*Valle del Software · Pruebas de Software · 7mo Semestre*